In [1]:
DATA_DIR = "dataset/"
MODEL_DIR = "model/"
CHECKPOINT_DIR = "epoch_model/"
RESULT_DIR = "result/"

RANDOM_STATE = 17

In [2]:
USERS_CSV = DATA_DIR + "users.csv"
TRAIN_CSV = DATA_DIR + "train.csv"
PREDICT_CSV = DATA_DIR + "test.csv"

# Define Model

In [3]:
MODEL_NAME = "sklearn_GradientBoostingClassifier_random17_testsize05"


from sklearn.ensemble import GradientBoostingClassifier


model = GradientBoostingClassifier()

# Import Data

In [4]:
import pandas


raw_users = pandas.read_csv(USERS_CSV)
raw_train = pandas.read_csv(TRAIN_CSV)
raw_predict = pandas.read_csv(PREDICT_CSV)

# Data Cleaning

In [5]:
from datetime import datetime


def get_unix(s):
    _temp = datetime.strptime(s[:-6], "%Y-%m-%d %H:%M:%S")
    return int((_temp - datetime(1970,1,1)).total_seconds())

In [6]:
def convert_nan(x):
    try:
        return int(x)
    except:
        return 1000 # impossible value

In [7]:
# references: https://www.kaggle.com/c/student-shopee-code-league-marketing-analytics/discussion/171531

def repair_noise(df):
    df["grass_date"] = df["grass_date"].apply(get_unix)

    df.loc[(df.last_open_day == "Never open") & df.open_count_last_10_days > 0, "last_open_day"] = 10
    df.loc[(df.last_open_day == "Never open") & df.open_count_last_30_days > 0, "last_open_day"] = 30
    df.loc[(df.last_open_day == "Never open") & df.open_count_last_60_days > 0, "last_open_day"] = 60

    df.loc[(df.last_login_day == "Never login") & df.login_count_last_10_days > 0, "last_login_day"] = 10
    df.loc[(df.last_login_day == "Never login") & df.login_count_last_30_days > 0, "last_login_day"] = 30
    df.loc[(df.last_login_day == "Never login") & df.login_count_last_60_days > 0, "last_login_day"] = 60

    df.loc[(df.last_checkout_day == "Never checkout") & df.checkout_count_last_10_days > 0, "last_checkout_day"] = 10
    df.loc[(df.last_checkout_day == "Never checkout") & df.checkout_count_last_30_days > 0, "last_checkout_day"] = 30
    df.loc[(df.last_checkout_day == "Never checkout") & df.checkout_count_last_60_days > 0, "last_checkout_day"] = 60

    df["last_open_day"] = df["last_open_day"].apply(convert_nan)
    df["last_login_day"] = df["last_login_day"].apply(convert_nan)
    df["last_checkout_day"] = df["last_checkout_day"].apply(convert_nan)

    return df

In [8]:
domain_map = {
    "@gmail.com"      : 0,
    "@yahoo.com"      : 1,
    "@hotmail.com"    : 2,
    "other"           : 3,
    "@icloud.com"     : 4,
    "@qq.com"         : 5,
    "@outlook.com"    : 6,
    "@ymail.com"      : 7,
    "@live.com"       : 8,
    "@163.com"        : 9,
    "@rocketmail.com" : 10,
}

# Data Processing

In [9]:
CATEGORICAL_DATA = [
    "country_code",
    "grass_date",
    "attr_1",
    "attr_3",
    "domain",
]

CONTINUOUS_DATA = [
#     "user_id",
    "subject_line_length",
    "last_open_day",
    "last_login_day",
    "last_checkout_day",
    "open_count_last_10_days",
    "open_count_last_30_days",
    "open_count_last_60_days",
    "login_count_last_10_days",
    "login_count_last_30_days",
    "login_count_last_60_days",
    "checkout_count_last_10_days",
    "checkout_count_last_30_days",
    "checkout_count_last_60_days",
    "age",
]

MISC_DATA = [
    "attr_2",
]

RESULT_INDEX = [
    "row_id",
]

RESULT_DATA = [
    "open_flag",
]

In [10]:
import numpy
from sklearn.preprocessing import StandardScaler


def normalize_scaler(a, b):
    scaler = StandardScaler()
    scaler.fit(a)
    a, b = scaler.transform(a), scaler.transform(b)
    return a, b

In [11]:
import numpy
from sklearn.preprocessing import OneHotEncoder


def normalize_ohe(a, b):
    ohe = OneHotEncoder(handle_unknown="ignore")
    ohe.fit(a)
    a, b = ohe.transform(a), ohe.transform(b)
    return pandas.DataFrame(a.toarray()), pandas.DataFrame(b.toarray())

In [12]:
def process_data(train, predict):
    for category in CONTINUOUS_DATA:
        train[category], predict[category] = normalize_scaler(
            train[[category]],
            predict[[category]],
        )
        
    train_categorical, predict_categorical = normalize_ohe(
        train[CATEGORICAL_DATA],
        predict[CATEGORICAL_DATA],
    )
    train = train.drop(columns=CATEGORICAL_DATA).join(train_categorical)
    predict = predict.drop(columns=CATEGORICAL_DATA).join(predict_categorical)

    return train, predict

# Do

In [13]:
users_data = raw_users.copy(deep=True)

users_data["domain"] = users_data["domain"].replace(domain_map)

users_data = users_data.apply(pandas.to_numeric, downcast="signed", errors="coerce")

users_data[["attr_1"]] = users_data[["attr_1"]].fillna(1)
users_data[["attr_2"]] = users_data[["attr_2"]].fillna(-1)
users_data[["attr_3"]] = users_data[["attr_3"]].fillna(-1)
users_data["age"] = users_data["age"].fillna(users_data["age"].mean())

In [14]:
train_data = repair_noise(pandas.merge(raw_train.copy(deep=True), users_data, on="user_id", how="left"))
predict_data = repair_noise(pandas.merge(raw_predict.copy(deep=True), users_data, on="user_id", how="left"))

In [15]:
train_data, predict_data = process_data(train_data, predict_data)

# Data Split

In [16]:
from sklearn.model_selection import train_test_split


X_train_dataset, X_test_dataset, y_train_dataset, y_test_dataset = train_test_split(
    train_data.loc[:, ~train_data.columns.isin(MISC_DATA + RESULT_INDEX + RESULT_DATA)],
    train_data[RESULT_DATA].values.reshape(-1),
    test_size=0.05,
    random_state=RANDOM_STATE,
)

# Train Model

In [17]:
model = model.fit(X_train_dataset.values, y_train_dataset)

# Accuracy Test

In [18]:
import numpy
from sklearn import metrics


y_pred = model.predict(X_test_dataset)

print("Acc : {0:.5f}".format(metrics.accuracy_score(y_test_dataset, y_pred)))
print("MAE : {0:.5f}".format(metrics.mean_absolute_error(y_test_dataset, y_pred)))
print("MSE : {0:.5f}".format(metrics.mean_squared_error(y_test_dataset, y_pred)))
print("RMSE: {0:.5f}".format(numpy.sqrt(metrics.mean_squared_error(y_test_dataset, y_pred))))
print("MCC : {0:.5f}".format(metrics.matthews_corrcoef(y_test_dataset, y_pred)))

Acc : 0.88986
MAE : 0.11014
MSE : 0.11014
RMSE: 0.33188
MCC : 0.54046


# Predict

In [19]:
predict_data["open_flag"] = model.predict(
    predict_data.loc[:, ~predict_data.columns.isin(MISC_DATA + RESULT_INDEX + RESULT_DATA)]
)

# Save Result

In [20]:
result = predict_data.copy(deep=True)
result = result[RESULT_INDEX + RESULT_DATA]

result.to_csv(RESULT_DIR + MODEL_NAME + "_result.csv", index=False)

# Compare

In [21]:
import glob


SUBMISSIONS = [
    filename[7:] for filename in glob.glob(
        RESULT_DIR + "s" + 8 * "[0-9]" + "_" + 5 * "[0-9]" + "*" + ".csv"
    )
]
SUBMISSIONS.sort(key=lambda x: x[10:15])

for submission in SUBMISSIONS:
    reta = pandas.read_csv(RESULT_DIR + submission)
    retb = pandas.read_csv(RESULT_DIR + MODEL_NAME + "_result.csv")
    rett = (reta["open_flag"] == retb["open_flag"]).value_counts(normalize=True).to_frame()
    
    print("{0:20s}: {1:9.5f} %".format(submission[:20], rett.loc[True, "open_flag"] * 100))

s16741387_49135.csv :  97.37538 %
s16821084_49288.csv :  97.87922 %
s16827813_49559.csv : 100.00000 %
s16822576_49597.csv :  98.07397 %
s16827570_49825.csv :  98.08647 %
s16742467_49869.csv :  96.95194 %
s16815547_49912.csv :  97.22173 %
s16819104_49931.csv :  97.94175 %
s16772608_49933.csv :  97.42898 %
s16772910_50084.csv :  97.26461 %
s16816045_50121.csv :  97.55405 %
s16742639_50279.csv :  97.28426 %
s16757603_50307.csv :  97.61658 %
s16770919_50425.csv :  98.14544 %
s16821141_50821_vote:  97.99535 %
